In [11]:
!pip -q install transformers sentence-transformers faiss-cpu pymupdf streamlit pyngrok accelerate sentencepiece

In [100]:
import re
import faiss
import fitz
import torch
import numpy as np
import transformers

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [13]:
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

transformers version: 5.0.0
torch version: 2.10.0+cu128
CUDA available: True
Using device: cuda


In [99]:
CONFIG = {
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "summarizer_model_name": "facebook/bart-large-cnn",
    "chunk_size": 120,
    "chunk_overlap": 30,
    "top_k": 6,
    "summary_max_length": 180,
    "summary_min_length": 60
}

In [101]:
embedder = SentenceTransformer(CONFIG["embedding_model_name"])

summarizer_tokenizer = AutoTokenizer.from_pretrained(CONFIG["summarizer_model_name"])
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["summarizer_model_name"])
summarizer_model = summarizer_model.to(device)

print("Models loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Models loaded successfully!


In [102]:
def bart_summarize_text(text, max_input_tokens=900, max_summary_tokens=180, min_summary_tokens=60):
    inputs = summarizer_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    ).to(device)

    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_summary_tokens,
        min_length=min_summary_tokens,
        num_beams=4,
        early_stopping=True
    )

    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [103]:
def extract_text_from_uploaded_pdf(uploaded_file):
    uploaded_file.seek(0)
    doc = fitz.open(stream=uploaded_file.read(), filetype="pdf")

    pages_data = []

    for page_num in range(len(doc)):
        page = doc[page_num]

        text = page.get_text("text")

        if not text.strip():
            blocks = page.get_text("blocks")
            text = " ".join(
                block[4].strip()
                for block in blocks
                if len(block) > 4 and block[4].strip()
            )

        text = re.sub(r"\s+", " ", text).strip()

        pages_data.append({
            "page_number": page_num + 1,
            "text": text
        })

    return pages_data

In [104]:
def build_full_text(pages_data):
    full_text = "\n\n".join(
        [f"[Page {item['page_number']}] {item['text']}" for item in pages_data if item["text"].strip()]
    )
    return full_text

In [105]:
def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words)
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [106]:
def create_chunk_embeddings(chunks, embedder):
    if not chunks:
        return np.array([])

    embeddings = embedder.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    return embeddings

In [107]:
def build_faiss_index(chunk_embeddings):
    if len(chunk_embeddings) == 0:
        raise ValueError("No embeddings were created. The PDF may contain no extractable text.")

    if len(chunk_embeddings.shape) != 2:
        raise ValueError(f"Embeddings shape is invalid: {chunk_embeddings.shape}")

    embedding_dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(chunk_embeddings)
    return index

In [108]:
def retrieve_relevant_chunks(query, embedder, index, chunks, top_k=6):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "chunk_id": int(idx),
            "distance": float(distances[0][rank]),
            "text": chunks[idx]
        })

    return results

In [109]:
def summarize_long_text(text, max_chunk_words=900):
    words = text.split()
    sub_texts = []

    start = 0
    while start < len(words):
        end = start + max_chunk_words
        sub_texts.append(" ".join(words[start:end]))
        start = end

    partial_summaries = []

    for i, sub in enumerate(sub_texts):
        print(f"Summarizing part {i+1}/{len(sub_texts)}...")
        summary = bart_summarize_text(
            sub,
            max_input_tokens=900,
            max_summary_tokens=CONFIG["summary_max_length"],
            min_summary_tokens=CONFIG["summary_min_length"]
        )
        partial_summaries.append(summary)

    if len(partial_summaries) == 1:
        return partial_summaries[0]

    combined_summary_text = " ".join(partial_summaries)

    final_summary = bart_summarize_text(
        combined_summary_text,
        max_input_tokens=900,
        max_summary_tokens=CONFIG["summary_max_length"],
        min_summary_tokens=CONFIG["summary_min_length"]
    )

    return final_summary

In [130]:
def answer_question_with_rag(question, embedder, index, chunks, tokenizer, model, top_k=6):
    retrieved = retrieve_relevant_chunks(question, embedder, index, chunks, top_k=top_k)
    context = "\n\n".join([item["text"] for item in retrieved])


    prompt = f"""
Summarize the answer to this question based on the context.

Context:
{context}

Question: {question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=120,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved
    }

In [116]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded file:", pdf_name)

Saving 15-3-Chapter 3.pdf to 15-3-Chapter 3.pdf
Uploaded file: 15-3-Chapter 3.pdf


In [119]:
uploaded_file = open(pdf_name, "rb")

In [120]:
pages_data = extract_text_from_uploaded_pdf(uploaded_file)

print("Total pages:", len(pages_data))
print("Non-empty pages:", sum(1 for p in pages_data if p["text"].strip()))
print("Average chars per page:", round(sum(len(p["text"]) for p in pages_data) / max(len(pages_data), 1), 2))

print("\nFirst page sample:\n")
print(pages_data[0]["text"][:1000])

Total pages: 88
Non-empty pages: 88
Average chars per page: 38.78

First page sample:

Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering 1


In [121]:
full_text = build_full_text(pages_data)

print("Total characters:", len(full_text))
print("\nSample from full text:\n")
print(full_text[:2000])

Total characters: 4458

Sample from full text:

[Page 1] Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering 1

[Page 2] Image Enhancement: • is the process of manipulating images so that the result is more suitable than the original for a specific application. As following: Highlighting interesting detail in images. Removing noise from images. Making images more visually appealing. • Enhancement are problem-oriented techniques. • Ex, the method for enhancing X-ray images may be not suitable for enhancing satellite images . 2

[Page 3] Spatial Domain Image Enhancement: • Most spatial domain enhancement operations can be reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some operator defined over some neighborhood of (x, y) 3

[Page 4] Spatial operations: • Spatial operations are performed directly on the pixels of a given image. Point : the output value at a specific coordinate is dependent o

In [122]:
chunks = chunk_text(
    full_text,
    chunk_size=CONFIG["chunk_size"],
    overlap=CONFIG["chunk_overlap"]
)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0][:1000])

Total chunks: 9

First chunk:

[Page 1] Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering 1 [Page 2] Image Enhancement: • is the process of manipulating images so that the result is more suitable than the original for a specific application. As following: Highlighting interesting detail in images. Removing noise from images. Making images more visually appealing. • Enhancement are problem-oriented techniques. • Ex, the method for enhancing X-ray images may be not suitable for enhancing satellite images . 2 [Page 3] Spatial Domain Image Enhancement: • Most spatial domain enhancement operations can be reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some


In [123]:
chunk_embeddings = create_chunk_embeddings(chunks, embedder)
print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (9, 384)


In [124]:
index = build_faiss_index(chunk_embeddings)

print("FAISS index built successfully!")
print("Total vectors in index:", index.ntotal)

FAISS index built successfully!
Total vectors in index: 9


In [125]:
test_query = "What is image enhancement?"
retrieved_chunks = retrieve_relevant_chunks(
    test_query,
    embedder,
    index,
    chunks,
    top_k=CONFIG["top_k"]
)

for item in retrieved_chunks:
    print("=" * 100)
    print(f"Rank: {item['rank']} | Chunk ID: {item['chunk_id']} | Distance: {item['distance']:.4f}")
    print(item["text"][:1200])
    print()

Rank: 1 | Chunk ID: 0 | Distance: 0.4547
[Page 1] Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering 1 [Page 2] Image Enhancement: • is the process of manipulating images so that the result is more suitable than the original for a specific application. As following: Highlighting interesting detail in images. Removing noise from images. Making images more visually appealing. • Enhancement are problem-oriented techniques. • Ex, the method for enhancing X-ray images may be not suitable for enhancing satellite images . 2 [Page 3] Spatial Domain Image Enhancement: • Most spatial domain enhancement operations can be reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some

Rank: 2 | Chunk ID: 1 | Distance: 1.1597
reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some operator defined over some neighborhood of (x, y) 3 [Page 4] Spat

In [126]:
sample_book_part = full_text[:4000]

sample_summary = bart_summarize_text(
    sample_book_part,
    max_input_tokens=900,
    max_summary_tokens=CONFIG["summary_max_length"],
    min_summary_tokens=CONFIG["summary_min_length"]
)

print("Sample Summary:\n")
print(sample_summary)

Sample Summary:

Image Enhancement is the process of manipulating images so that the result is more suitable than the original for a specific application. Spatial operations are performed directly on the pixels of a given image. Point processing operations take the form s = T ( r ) where s refers to the processed image pixel value. Power-law curves with fractional values of γ map a narrow range of dark images into a wider range of light images.


In [127]:
result = answer_question_with_rag(
    "What is image enhancement?",
    embedder,
    index,
    chunks,
    summarizer_tokenizer,
    summarizer_model,
    top_k=CONFIG["top_k"]
)

print("Question:", result["question"])
print("Answer:", result["answer"])

Question: What is image enhancement?
Answer: Use the context below to answer the question. Give a short and clear answer in 2-3 sentences maximum. Do NOT include unnecessary details. If possible, mention the page number. Context: Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering. Image Enhancement: Is the process of manipulating images so that the result is more suitable than the original.


In [128]:
for item in result["retrieved_chunks"]:
    print("=" * 100)
    print(f"Rank: {item['rank']} | Chunk ID: {item['chunk_id']} | Distance: {item['distance']:.4f}")
    print(item["text"][:1500])
    print()

Rank: 1 | Chunk ID: 0 | Distance: 0.4547
[Page 1] Computer Vision Chapter 3 Intensity Transformation and Spatial Filtering 1 [Page 2] Image Enhancement: • is the process of manipulating images so that the result is more suitable than the original for a specific application. As following: Highlighting interesting detail in images. Removing noise from images. Making images more visually appealing. • Enhancement are problem-oriented techniques. • Ex, the method for enhancing X-ray images may be not suitable for enhancing satellite images . 2 [Page 3] Spatial Domain Image Enhancement: • Most spatial domain enhancement operations can be reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some

Rank: 2 | Chunk ID: 1 | Distance: 1.1597
reduced to the form g (x, y) = T[ f (x, y)] • where f (x, y)is the input image, g (x, y)is the processed image and T is some operator defined over some neighborhood of (x, y) 3 [Page 4] Spat

In [129]:
%%writefile app.py
import re
import faiss
import fitz
import torch
import numpy as np
import streamlit as st
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CONFIG = {
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "summarizer_model_name": "facebook/bart-large-cnn",
    "chunk_size": 120,
    "chunk_overlap": 30,
    "top_k": 6,
    "summary_max_length": 180,
    "summary_min_length": 60
}

device = "cuda" if torch.cuda.is_available() else "cpu"

@st.cache_resource
def load_models():
    embedder = SentenceTransformer(CONFIG["embedding_model_name"])
    summarizer_tokenizer = AutoTokenizer.from_pretrained(CONFIG["summarizer_model_name"])
    summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["summarizer_model_name"])
    summarizer_model = summarizer_model.to(device)
    return embedder, summarizer_tokenizer, summarizer_model

def bart_summarize_text(text, tokenizer, model, max_input_tokens=900, max_summary_tokens=180, min_summary_tokens=60):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_summary_tokens,
        min_length=min_summary_tokens,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

def extract_text_from_uploaded_pdf(uploaded_file):
    uploaded_file.seek(0)
    doc = fitz.open(stream=uploaded_file.read(), filetype="pdf")

    pages_data = []
    for page_num in range(len(doc)):
        page = doc[page_num]

        text = page.get_text("text")

        if not text.strip():
            blocks = page.get_text("blocks")
            text = " ".join(
                block[4].strip()
                for block in blocks
                if len(block) > 4 and block[4].strip()
            )

        text = re.sub(r"\s+", " ", text).strip()

        pages_data.append({
            "page_number": page_num + 1,
            "text": text
        })

    return pages_data

def build_full_text(pages_data):
    full_text = "\n\n".join(
        [f"[Page {item['page_number']}] {item['text']}" for item in pages_data if item["text"].strip()]
    )
    return full_text

def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words)
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

def create_chunk_embeddings(chunks, embedder):
    if not chunks:
        return np.array([])

    embeddings = embedder.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    return embeddings

def build_faiss_index(chunk_embeddings):
    if len(chunk_embeddings) == 0:
        raise ValueError("No embeddings were created. The PDF may contain no extractable text.")

    if len(chunk_embeddings.shape) != 2:
        raise ValueError(f"Embeddings shape is invalid: {chunk_embeddings.shape}")

    embedding_dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(chunk_embeddings)
    return index

def retrieve_relevant_chunks(query, embedder, index, chunks, top_k=6):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "chunk_id": int(idx),
            "distance": float(distances[0][rank]),
            "text": chunks[idx]
        })

    return results

def summarize_long_text(text, tokenizer, model, max_chunk_words=900):
    words = text.split()
    sub_texts = []

    start = 0
    while start < len(words):
        end = start + max_chunk_words
        sub_texts.append(" ".join(words[start:end]))
        start = end

    partial_summaries = []

    for sub in sub_texts:
        summary = bart_summarize_text(
            sub,
            tokenizer,
            model,
            max_input_tokens=900,
            max_summary_tokens=CONFIG["summary_max_length"],
            min_summary_tokens=CONFIG["summary_min_length"]
        )
        partial_summaries.append(summary)

    if len(partial_summaries) == 1:
        return partial_summaries[0]

    combined_summary_text = " ".join(partial_summaries)

    final_summary = bart_summarize_text(
        combined_summary_text,
        tokenizer,
        model,
        max_input_tokens=900,
        max_summary_tokens=CONFIG["summary_max_length"],
        min_summary_tokens=CONFIG["summary_min_length"]
    )

    return final_summary

def answer_question_with_rag(question, embedder, index, chunks, tokenizer, model, top_k=6):
    retrieved = retrieve_relevant_chunks(question, embedder, index, chunks, top_k=top_k)
    context = "\n\n".join([item["text"] for item in retrieved])


    prompt = f"""
Summarize the answer to this question based on the context.

Context:
{context}

Question: {question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=120,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved
    }

st.set_page_config(page_title="Book Assistant", layout="wide")
st.title("📘 Book Assistant with BART + RAG")

embedder, summarizer_tokenizer, summarizer_model = load_models()

uploaded_file = st.file_uploader("Upload a PDF book", type=["pdf"])

if uploaded_file is not None:
    with st.spinner("Processing book..."):
        pages_data = extract_text_from_uploaded_pdf(uploaded_file)
        full_text = build_full_text(pages_data)

        st.write("Detected pages:", len(pages_data))
        st.write("Extracted characters:", len(full_text))

        non_empty_pages = sum(1 for p in pages_data if p["text"].strip())
        avg_chars = sum(len(p["text"]) for p in pages_data) / max(len(pages_data), 1)

        st.write("Non-empty pages:", non_empty_pages)
        st.write("Average chars per page:", round(avg_chars, 2))

        if not full_text.strip():
            st.error("No extractable text was found in this PDF. It may be a scanned or image-based PDF.")
            st.stop()

        chunks = chunk_text(full_text, CONFIG["chunk_size"], CONFIG["chunk_overlap"])
        st.write("Total chunks:", len(chunks))

        if len(chunks) == 0:
            st.error("Text extraction worked, but no chunks were created.")
            st.stop()

        chunk_embeddings = create_chunk_embeddings(chunks, embedder)

        if len(chunk_embeddings) == 0:
            st.error("No embeddings were created from the extracted text.")
            st.stop()

        index = build_faiss_index(chunk_embeddings)

    st.success(f"Book processed successfully! Pages: {len(pages_data)} | Chunks: {len(chunks)}")

    tab1, tab2 = st.tabs(["Summary", "Ask Questions"])

    with tab1:
        if st.button("Generate Summary"):
            with st.spinner("Summarizing..."):
                summary = summarize_long_text(full_text[:15000], summarizer_tokenizer, summarizer_model)
            st.subheader("Book Summary")
            st.write(summary)

    with tab2:
        question = st.text_input("Ask a question about the book")
        if st.button("Get Answer"):
            if question.strip():
                with st.spinner("Searching and generating answer..."):
                    result = answer_question_with_rag(
                        question,
                        embedder,
                        index,
                        chunks,
                        summarizer_tokenizer,
                        summarizer_model,
                        top_k=CONFIG["top_k"]
                    )

                st.subheader("Answer")
                st.write(result["answer"])
                st.caption("Answer generated from retrieved context")

                with st.expander("Retrieved Chunks"):
                    for item in result["retrieved_chunks"]:
                        st.markdown(
                            f"**Rank:** {item['rank']} | **Chunk ID:** {item['chunk_id']} | **Distance:** {item['distance']:.4f}"
                        )
                        st.write(item["text"][:1500])
                        st.markdown("---")

Overwriting app.py


In [97]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [90]:
!ngrok config add-authtoken 3CitZQlGWPUHpArF1aToTWDa9O6_3NJ2sJvuDiGfuXuZ5T38v

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [98]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://extrude-ageless-pension.ngrok-free.dev" -> "http://localhost:8501"


In [96]:
!pkill streamlit
!pkill ngrok

In [94]:
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

ngrok.kill()
print("All ngrok tunnels closed.")

All ngrok tunnels closed.
